# 01. Exploratory Data Analysis & Market Signal Extraction

**Research Paper:** *A Deep Learning Framework for Predictive Feature Prioritization in Early-Stage Software Startups: Integrating Historical Delivery Data and Market Signals* (MDPI Computers 2026, 15, 380)  
**Dataset:** LinkedIn Job Postings (2023–2024), Kaggle dataset by arshkon (`postings.csv`)

---

## 1. Research Problem Formulation & Objectives
In early-stage software startups, product development is constrained by limited engineering bandwidth and high market uncertainty. The research paper proposes prioritizing software backlog features by combining:
1. **Internal Historical Delivery Velocity** (sprint records, cycle times)
2. **External Market Signals** (job market demand signals extracted from public postings)

In this notebook, we perform an exhaustive exploratory data analysis (EDA) of the 123,849 LinkedIn job postings to inspect schema, data distributions, missingness patterns, temporal coverage, and formulation of the market-signal target proxy.


In [1]:
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path
sys.path.append("..")
from src import config, preprocessing, visualization

print("Libraries imported successfully.")


Libraries imported successfully.


## 2. Dataset Ingestion & Schema Inspection
We load the raw postings dataset from `Dataset/postings.csv` and inspect its shape and column definitions.


In [2]:
# Inspect precomputed dataset summary
summary_path = Path("../Results/dataset_inspection_summary.json")
if summary_path.exists():
    with open(summary_path, "r") as f:
        summary = json.load(f)
    print(f"Total Rows: {summary['total_rows']:,}")
    print(f"Total Columns: {summary['total_columns']}")
    print(f"Columns: {summary['columns']}")
else:
    print("Run Dataset/inspect_data.py to generate dataset summary.")


Total Rows: 123,849
Total Columns: 31
Columns: ['job_id', 'company_name', 'title', 'description', 'max_salary', 'pay_period', 'location', 'company_id', 'views', 'med_salary', 'min_salary', 'formatted_work_type', 'applies', 'original_listed_time', 'remote_allowed', 'job_posting_url', 'application_url', 'application_type', 'expiry', 'closed_time', 'formatted_experience_level', 'skills_desc', 'listed_time', 'posting_domain', 'sponsored', 'work_type', 'currency', 'compensation_type', 'normalized_salary', 'zip_code', 'fips']


## 3. Missing Value Analysis
Understanding missing values is vital to prevent unintended data deletion or biased imputation.


In [3]:
# Display top missing columns
if summary_path.exists():
    missing_df = pd.DataFrame([
        {"Column": col, "Missing_Count": cnt, "Missing_Pct": (cnt / summary["total_rows"]) * 100}
        for col, cnt in summary["missing_counts"].items()
    ]).sort_values(by="Missing_Pct", ascending=False)
    print("Top 15 Columns with Missing Data:")
    print(missing_df.head(15).to_string(index=False))


Top 15 Columns with Missing Data:
                    Column  Missing_Count  Missing_Pct
               closed_time         122776    99.133622
               skills_desc         121410    98.030666
                med_salary         117569    94.929309
            remote_allowed         108603    87.689848
                   applies         100529    81.170619
                min_salary          94056    75.944093
                max_salary          94056    75.944093
                  currency          87776    70.873402
         normalized_salary          87776    70.873402
                pay_period          87776    70.873402
         compensation_type          87776    70.873402
            posting_domain          39968    32.271556
           application_url          36665    29.604599
formatted_experience_level          29409    23.745852
                      fips          27415    22.135827


## 4. Formulation of the Target Variable (`formatted_experience_level`)
The binary market-signal target is defined as:
- **$y = 1$**: `Entry level` (Proxy for Low Market Priority / Lower Baseline Seniority Demand)
- **$y = 0$**: Other valid experience levels (`Mid-Senior level`, `Associate`, `Director`, `Executive`, `Internship`)

### Methodological Rule:
Rows with missing `formatted_experience_level` ($29,409$ rows, $23.75\%$) are explicitly excluded rather than silently imputed to either class to avoid target corruption.


In [4]:
if summary_path.exists():
    print(f"Filtered Valid Rows: {summary['valid_total']:,} ({summary['valid_total']/summary['total_rows']*100:.2f}%)")
    print(f"Class 1 (Entry level / Low Priority Proxy): {summary['entry_count']:,} ({summary['entry_count']/summary['valid_total']*100:.2f}%)")
    print(f"Class 0 (Other Levels / High Priority Proxy): {summary['other_count']:,} ({summary['other_count']/summary['valid_total']*100:.2f}%)")


Filtered Valid Rows: 94,440 (76.25%)
Class 1 (Entry level / Low Priority Proxy): 36,708 (38.87%)
Class 0 (Other Levels / High Priority Proxy): 57,732 (61.13%)


## 5. Temporal Coverage Validation
Timestamps (`listed_time`) span from March 24, 2024 to April 20, 2024. This reflects a dense window of active hiring signals in tech.


In [5]:
if summary_path.exists():
    print(f"Temporal Window: {summary['min_date']} to {summary['max_date']}")


Temporal Window: 2024-03-24 to 2024-04-20


## 6. Key Findings & Research Limitations
1. **External Proxy:** `formatted_experience_level` is an external market demand proxy, not a direct measurement of feature priority in a Jira backlog.
2. **Missing Sprint Data:** The paper's private startup sprint delivery logs are unavailable. We implement the market-signal 1D-CNN branch using the verified LinkedIn postings.
3. **Data Leakage Prevention:** `formatted_experience_level` must be strictly excluded from model feature inputs.
